In [112]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import statsmodels.api as sm
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMRegressor
#这里改了一下
#这里又修改了一下，现在要把这条上传到github了，之后去网站检查一下看有没有。

In [113]:
# 数据预处理
df = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_with_IMR_rename.csv')

required_cols = ['ExcellenceRate','PassRate', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays',
                 'SchType', 'SchLocation', 'Gender', 'IMR','Grade', 'District', 'Date', 'SchYear']
df = df.dropna(subset=required_cols)

# 核心自变量归一化
scaler = StandardScaler()
for var in ['r1h', 'r3h', 'FluctMonth']:
    ##这里又改了一下
    df[f'Norm{var}'] = scaler.fit_transform(df[[var]])

df_ml = df.copy()
# 固定效应转哑变量 (W_cols)
df_ml = pd.get_dummies(df_ml, columns=['Grade','District','Date','SchYear'], drop_first=True)   #'SchYear'

# 因变量为PassRate

In [114]:
# 变量分类定义
Y_col = 'PassRate'
T_cols = ['Normr1h', 'Normr3h', 'NormFluctMonth']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在分析核心变量: {t_var} {'='*20}")
    
    if t_var=='Normr1h':
        W_cols=['IMR','Normr3h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'Date_'])] #, 'Date_', 'SchYear_'      最前面的框中去掉： ,'Normr3h','NormFluctMonth'
    elif t_var=='Normr3h':
        W_cols=['IMR','Normr1h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'Date_'])] #, 'Date_', 'SchYear_'  最前面的框中去掉： ,'Normr1h','NormFluctMonth'
    else:
        W_cols=['IMR','Normr1h', 'Normr3h'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'Date_'])] #, 'Date_', 'SchYear_'   最前面的框中去掉：,'Normr1h', 'Normr3h'


    # A. 构建并拟合因果森林
    seed = 32 #@@@@@
    est = CausalForestDML(
        model_y=LassoCV(random_state=seed),
        model_t=LassoCV(random_state=seed),
    
        discrete_treatment=False,
        n_estimators=2000, #@@@@@
        min_samples_leaf=100, #@@@@@
        min_var_fraction_leaf=0.1, #@@@@@
        cv=3,
        random_state=seed,
        n_jobs=-1 # 使用所有CPU核心加速
    )
    

    # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # 自动计算平衡权重：样本越少的组合，权重越高
    # compute_sample_weight 会确保所有类别的权重相等，会给样本少的组合（譬如1-0-1）更高权重，给样本多的组合(譬如0-1-1)更低权重。
    sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后再用X各个特征分开树节点，使得不同分叉的残差Y与残差T间的系数差别最大；
    # 然后求形成的树的叶节点的各个样本的残差Y与残差T间的系数的加权平均值，再把森林中各个树的该值平均，这样能得到
    # 最终在条件X下影响系数的平均值
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 平均处理效应推断 (ATE)
    ate_inf = est.ate_inference(X=df_ml[X_cols])
    print("\n--- 0. 平均处理效应 (ATE) ---")
    print("平均处理效应为：",ate_inf.mean_point)
    print("平均处理效应的p值为：",ate_inf.pvalue())
    print(" ")

    # C. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # D. 个体推断：显著效应占比   因为做了不同组合样本的等权，所以这个指标意义不大
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"--- 1. 显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    #这个不重要，永远不要了
    # # E. 调节效应显著性检验 (OLS 汇报)
    # print("\n--- 2. 调节效应显著性检验 (线性估计 OLS Summary) ---")
    # X_reg = sm.add_constant(df_ml[X_cols].astype(float))
    # ols_res = sm.OLS(hte_values, X_reg).fit()
    # print(ols_res.summary())


    # F. 各变量贡献度 (特征重要性)
    print("\n--- 3. 变量对效应异质性的贡献度 (Feature Importance) ---")
    for name, imp in zip(X_cols, est.feature_importances_):
        print(f"    {name}: {imp:.4f}")

    # G. 8种具体组合的预测效应
    print("\n--- 4. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        #return f"{st}-{sl}" 
        return f"{st}-{sl}-{ge}"
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    group_share = df_ml.groupby(X_cols).size().reset_index(name='Group_Count')
    group_share['Sample_Share'] = (group_share['Group_Count'] / len(df_ml) * 100).map(lambda x: f"{x:.2f}%")
    df_comb = df_comb.merge(group_share[X_cols + ['Sample_Share']], on=X_cols, how='left')
    print(df_comb[['Group_Desc', 'Sample_Share', 'treatment_effect', 'p_value']])

    # H. 分组汇总表生成 (用于导出)
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))

# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_PassRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在分析核心变量: Normr1h ====================

--- 0. 平均处理效应 (ATE) ---
平均处理效应为： -0.010604794421326495
平均处理效应的p值为： 0.09880475481198542
 
--- 1. 显著效应样本占比 (p < 0.05): 61.24%

--- 3. 变量对效应异质性的贡献度 (Feature Importance) ---
    SchType: 0.5268
    SchLocation: 0.2944
    Gender: 0.1788

--- 4. 各变量组合的具体效应预测 ---
  Group_Desc Sample_Share  treatment_effect   p_value
0   公立-农村-女生       28.67%         -0.015127  0.006356
1   公立-农村-男生       29.86%         -0.018922  0.002207
2   公立-城市-女生       17.58%         -0.003955  0.540856
3   公立-城市-男生       17.57%         -0.006901  0.303919
4   私立-农村-女生        0.38%          0.013049  0.209018
5   私立-农村-男生        0.48%          0.019907  0.077312
6   私立-城市-女生        2.74%          0.016727  0.078718
7   私立-城市-男生        2.72%          0.025182  0.005930

==================== 正在分析核心变量: Normr3h ====================

--- 0. 平均处理效应 (ATE) ---
平均处理效应为： 0.01566482432103642
平均处理效应的p值为： 0.06545318746576759
 
--- 1. 显著效应样本占比 (p < 0.05): 76.09%

--- 3. 变量

# 因变量为ExcellenceRate

In [115]:
# 变量分类定义
Y_col = 'ExcellenceRate'
T_cols = ['Normr1h', 'Normr3h', 'NormFluctMonth']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在分析核心变量: {t_var} {'='*20}")
    
    if t_var=='Normr1h':
        W_cols=['IMR','Normr3h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'Date_'])] #, 'Date_', 'SchYear_'   最前面的框中去掉：,'Normr3h','NormFluctMonth'
    elif t_var=='Normr3h':
        W_cols=['IMR','Normr1h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'Date_'])] #, 'Date_', 'SchYear_'   最前面的框中去掉： ,'Normr1h','NormFluctMonth'
    else:
        W_cols=['IMR','Normr1h', 'Normr3h'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'Date_'])] #, 'Date_', 'SchYear_'   最前面的框中去掉：,'Normr1h', 'Normr3h'

    # A. 构建并拟合因果森林
    seed = 32 #@@@@@
    est = CausalForestDML(
        model_y=LassoCV(random_state=seed),
        model_t=LassoCV(random_state=seed),
    
        discrete_treatment=False,
        n_estimators=2000, #@@@@@
        min_samples_leaf=100, #@@@@@
        min_var_fraction_leaf=0.1, #@@@@@
        cv=3,
        random_state=seed,
        n_jobs=-1 # 使用所有CPU核心加速
    )
    

    # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # 自动计算平衡权重：样本越少的组合，权重越高
    # compute_sample_weight 会确保所有类别的权重总和相等
    sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后再用X各个特征分开树节点，使得不同分叉的残差Y与残差T间的系数差别最大；
    # 然后求形成的树的叶节点的各个样本的残差Y与残差T间的系数的加权平均值，再把森林中各个树的该值平均，这样能得到
    # 最终在条件X下影响系数的平均值
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 平均处理效应推断 (ATE)
    ate_inf = est.ate_inference(X=df_ml[X_cols])
    print("\n--- 0. 平均处理效应 (ATE) ---")
    print("平均处理效应为：",ate_inf.mean_point)
    print("平均处理效应的p值为：",ate_inf.pvalue())


    # C. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # D. 个体推断：显著效应占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"--- 1. 显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    #这个不重要，永远不要了
    # # E. 调节效应显著性检验 (OLS 汇报)
    # print("\n--- 2. 调节效应显著性检验 (线性估计 OLS Summary) ---")
    # X_reg = sm.add_constant(df_ml[X_cols].astype(float))
    # ols_res = sm.OLS(hte_values, X_reg).fit()
    # print(ols_res.summary())


    # F. 各变量贡献度 (特征重要性)
    print("\n--- 3. 变量对效应异质性的贡献度 (Feature Importance) ---")
    for name, imp in zip(X_cols, est.feature_importances_):
        print(f"    {name}: {imp:.4f}")

    # G. 8种具体组合的预测效应
    print("\n--- 4. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    group_share = df_ml.groupby(X_cols).size().reset_index(name='Group_Count')
    group_share['Sample_Share'] = (group_share['Group_Count'] / len(df_ml) * 100).map(lambda x: f"{x:.2f}%")
    df_comb = df_comb.merge(group_share[X_cols + ['Sample_Share']], on=X_cols, how='left')
    print(df_comb[['Group_Desc', 'Sample_Share', 'treatment_effect', 'p_value']])

    # H. 分组汇总表生成 (用于导出)
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))

# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_ExcellenceRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在分析核心变量: Normr1h ====================

--- 0. 平均处理效应 (ATE) ---
平均处理效应为： -0.017332214846530646
平均处理效应的p值为： 0.016018352502540013
--- 1. 显著效应样本占比 (p < 0.05): 64.85%

--- 3. 变量对效应异质性的贡献度 (Feature Importance) ---
    SchType: 0.3696
    SchLocation: 0.3130
    Gender: 0.3174

--- 4. 各变量组合的具体效应预测 ---
  Group_Desc Sample_Share  treatment_effect   p_value
0   公立-农村-女生       28.67%         -0.020895  0.000527
1   公立-农村-男生       29.86%         -0.021454  0.000405
2   公立-城市-女生       17.58%         -0.006130  0.438945
3   公立-城市-男生       17.57%         -0.008652  0.282014
4   私立-农村-女生        0.38%         -0.034868  0.000951
5   私立-农村-男生        0.48%         -0.027849  0.008943
6   私立-城市-女生        2.74%         -0.041140  0.000360
7   私立-城市-男生        2.72%         -0.034682  0.001864

==================== 正在分析核心变量: Normr3h ====================

--- 0. 平均处理效应 (ATE) ---
平均处理效应为： 0.018998725036684688
平均处理效应的p值为： 0.03951983787435149
--- 1. 显著效应样本占比 (p < 0.05): 81.93%

--- 3. 变量对效